# 03 — Business Analysis

Calculates the core KPIs and answers the five initial business questions, using the
cleaned tables built in Phase 3 (`orders_analytics`, `order_items_analytics`) via
`src/metrics.py` (Python) and `sql/*.sql` (DuckDB). No dashboard styling here —
this is the analytical layer the dashboard will later present.

**Population:** all headline metrics use delivered orders only. See README.md
("Cleaning & Analytical Tables") for the full population and grain definitions.

In [1]:
import sys
sys.path.insert(0, "..")

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

from src import metrics as m
from src.database import get_connection

con = get_connection()
orders_analytics = con.execute("SELECT * FROM orders_analytics").fetchdf()
order_items_analytics = con.execute("SELECT * FROM order_items_analytics").fetchdf()

delivered_items = order_items_analytics[order_items_analytics["is_delivered"]]
print(f"orders_analytics: {len(orders_analytics):,} rows | order_items_analytics: {len(order_items_analytics):,} rows")

orders_analytics: 99,441 rows | order_items_analytics: 112,650 rows


## Executive KPIs

In [2]:
kpis = {
    "Revenue (delivered)": f"R$ {m.total_revenue(orders_analytics):,.2f}",
    "Delivered orders": f"{m.order_count(orders_analytics):,}",
    "Unique customers": f"{m.unique_customer_count(orders_analytics):,}",
    "Average order value": f"R$ {m.average_order_value(orders_analytics):,.2f}",
    "Repeat customer rate": f"{m.repeat_customer_rate(orders_analytics) * 100:.1f}%",
    "Average review score": f"{m.average_review_score(orders_analytics):.2f} / 5",
    "Reviewed-order rate": f"{m.reviewed_order_rate(orders_analytics) * 100:.1f}%",
    "On-time delivery rate": f"{m.on_time_delivery_rate(orders_analytics) * 100:.1f}%",
    "Average delivery time": f"{m.average_delivery_days(orders_analytics):.1f} days",
}
pd.Series(kpis, name="value").to_frame()

,value
Revenue (delivered),"R$ 13,221,498.11"
Delivered orders,"96,478"
Unique customers,"93,358"
Average order value,R$ 137.04
Repeat customer rate,3.0%
Average review score,4.16 / 5
Reviewed-order rate,99.3%
On-time delivery rate,93.2%
Average delivery time,12.1 days


**Cross-validation:** `sql/01_revenue_analysis.sql` query 1, `sql/03_retention_analysis.sql` query 1, and `sql/04_delivery_analysis.sql` query 1 independently recompute revenue/AOV, repeat rate, and on-time rate directly in DuckDB SQL. All match the Python results above exactly (see Phase 4 report) — the two implementations agree.

## Revenue Trends

Monthly revenue, order volume, and AOV. The Olist marketplace had negligible volume before 2017 (4 orders in Sep 2016, 1 delivered), and the dataset was extracted before orders placed after ~Aug 2018 had time to complete delivery — so **2016-09, 2016-10, 2016-12, and any 2018-09/2018-10 rows are not shown/compared as full months.**

In [3]:
monthly = m.monthly_summary(orders_analytics)

# flag partial periods explicitly rather than silently including them
partial_months = {"2016-09", "2016-10", "2016-12"}
monthly["period"] = np.where(monthly["month"].isin(partial_months), "partial", "full")

fig = px.bar(monthly, x="month", y="revenue", color="period",
             color_discrete_map={"full": "#2E5EAA", "partial": "#C9D6EA"},
             title="Monthly delivered-order revenue")
fig.update_layout(xaxis_tickangle=-45, showlegend=True)
fig.show()

In [4]:
fig = px.line(monthly[monthly["period"] == "full"], x="month", y="aov",
              title="Monthly AOV (full months only)", markers=True)
fig.update_layout(xaxis_tickangle=-45, yaxis_title="AOV (R$)")
fig.show()

**Finding:** revenue grew steadily from platform launch through late 2017, then plateaued at roughly R$0.85–0.98M/month for most of 2018 — order volume, not AOV, drives the trend (AOV stays in a narrow R$124–152 band throughout). November 2017 is a clear outlier (R$987.8k, +35% over October) consistent with Black Friday.

## Product Categories

In [5]:
cat_rev = (
    delivered_items.groupby("product_category_english")
    .agg(revenue=("price", "sum"), item_count=("order_item_id", "count"),
         order_count=("order_id", "nunique"), avg_item_price=("price", "mean"))
    .sort_values("revenue", ascending=False)
)
cat_rev["pct_of_revenue"] = cat_rev["revenue"] / cat_rev["revenue"].sum() * 100

# reconciliation check: category revenue must sum to the same total as orders_analytics
assert abs(cat_rev["revenue"].sum() - m.total_revenue(orders_analytics)) < 0.01

top15 = cat_rev.head(15).reset_index()
fig = px.bar(top15, x="revenue", y="product_category_english", orientation="h",
             title="Top 15 categories by delivered-order revenue")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, xaxis_title="Revenue (R$)", yaxis_title="")
fig.show()
top15[["product_category_english", "revenue", "order_count", "avg_item_price", "pct_of_revenue"]].round(2)

,product_category_english,revenue,order_count,avg_item_price,pct_of_revenue
0,health_beauty,1233131.72,8647,130.28,9.33
1,watches_gifts,1166176.98,5495,199.04,8.82
2,bed_bath_table,1023434.76,9272,93.44,7.74
3,sports_leisure,954852.55,7530,113.25,7.22
4,computers_accessories,888724.61,6530,116.26,6.72
5,furniture_decor,711927.69,6307,87.25,5.38
6,housewares,615628.69,5743,90.60,4.66
7,cool_stuff,610204.10,3559,164.12,4.62
8,auto,578966.65,3810,139.85,4.38
9,toys,471286.48,3804,116.94,3.56


**Note on order_count here:** because a single order can span multiple categories, these `order_count` values must not be summed and reported as "total orders" — they will exceed the true 96,478 delivered orders.

**Finding:** revenue is broadly distributed — the top category (`health_beauty`) is only 9.3% of revenue, and the top 10 categories together account for roughly two-thirds of it. No single category dominates the way it might in a more specialized retailer.

## Customer Retention

In [6]:
counts = m.customer_order_counts(orders_analytics)
freq = counts["delivered_order_count"].value_counts().sort_index()
print(freq.head(10))

single_rev = counts.loc[counts["delivered_order_count"] == 1, "total_revenue"].sum()
repeat_rev = counts.loc[counts["delivered_order_count"] >= 2, "total_revenue"].sum()
print(f"\nRevenue from single-purchase customers: R$ {single_rev:,.2f} ({single_rev/(single_rev+repeat_rev)*100:.1f}%)")
print(f"Revenue from repeat customers:          R$ {repeat_rev:,.2f} ({repeat_rev/(single_rev+repeat_rev)*100:.1f}%)")

delivered_order_count
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64

Revenue from single-purchase customers: R$ 12,493,089.36 (94.5%)
Revenue from repeat customers:          R$ 728,408.75 (5.5%)


In [7]:
fig = px.bar(x=freq.index.astype(str), y=freq.values, log_y=True,
             title="Delivered orders per customer (log scale)",
             labels={"x": "delivered orders", "y": "customers"})
fig.show()

**Finding: repeat purchasing is low.** Only 3.0% of customers placed more than one delivered order, and repeat customers contribute only 5.5% of revenue (computed above). This is consistent with `sql/03_retention_analysis.sql`, which independently confirms the same rate in SQL. Note this reflects the ~2-year dataset window — some "single-purchase" customers may have returned after the data cutoff.

`sql/03_retention_analysis.sql` (query 3) also shows repeat customers take a median of 29 days to place their second order, but a mean of over 80 days — a right-skewed distribution with a long tail of slow returners.

## Delivery Performance

In [8]:
eligible = m.delivery_timing_eligible(orders_analytics)
print(f"Timing-eligible delivered orders: {len(eligible):,} of {m.order_count(orders_analytics):,} delivered "
      f"({len(eligible)/m.order_count(orders_analytics)*100:.1f}%)")
print(f"On-time rate: {m.on_time_delivery_rate(orders_analytics)*100:.2f}%")
print(f"Late rate: {m.late_delivery_rate(orders_analytics)*100:.2f}%")
print(f"Average delivery time: {m.average_delivery_days(orders_analytics):.1f} days (median {m.median_delivery_days(orders_analytics):.0f})")

fig = px.histogram(eligible, x="delivery_delay_days", nbins=60,
                    title="Delivery delay distribution (days; negative = early)")
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(xaxis_range=[-40, 40], xaxis_title="delivery_delay_days (actual - estimated)")
fig.show()

Timing-eligible delivered orders: 96,470 of 96,478 delivered (100.0%)
On-time rate: 93.23%
Late rate: 6.77%
Average delivery time: 12.1 days (median 10)


**Finding:** delivery performance is strong overall — 93.2% of orders arrive on or before the estimated date, with a median delivery time of 10 days. The delay distribution is left-skewed: most orders arrive well before the estimate, and lateness is a long right tail rather than the norm.

## Customer Reviews vs. Delivery Delay

In [9]:
eligible = eligible.copy()
conditions = [
    eligible["delivery_delay_days"] < -7,
    eligible["delivery_delay_days"] <= 0,
    eligible["delivery_delay_days"] <= 3,
    eligible["delivery_delay_days"] <= 7,
]
choices = ["1. early by >7 days", "2. on-time / early (<=7d early)", "3. 1-3 days late", "4. 4-7 days late"]
eligible["delay_bucket"] = np.select(conditions, choices, default="5. 8+ days late")

bucket_summary = (
    eligible.groupby("delay_bucket")
    .agg(orders=("order_id", "nunique"), avg_review_score=("avg_review_score", "mean"))
    .reset_index()
)
fig = px.bar(bucket_summary, x="delay_bucket", y="avg_review_score",
             title="Average review score by delivery-delay bucket", text_auto=".2f")
fig.update_layout(yaxis_range=[1, 5], xaxis_title="", yaxis_title="avg review score")
fig.show()
bucket_summary

,delay_bucket,orders,avg_review_score
0,1. early by >7 days,71303,4.316995
1,2. on-time / early (<=7d early),18633,4.189424
2,3. 1-3 days late,1870,3.291037
3,4. 4-7 days late,1802,2.105549
4,5. 8+ days late,2862,1.697591


In [10]:
on_time_score = eligible.loc[eligible["delivery_delay_days"] <= 0, "avg_review_score"].mean()
late_score = eligible.loc[eligible["delivery_delay_days"] > 0, "avg_review_score"].mean()
print(f"Avg review score, on-time orders: {on_time_score:.2f}")
print(f"Avg review score, late orders:    {late_score:.2f}")

Avg review score, on-time orders: 4.29
Avg review score, late orders:    2.27


**Finding: late deliveries are associated with markedly lower review scores.** On-time/early orders average 4.29/5, versus 2.27/5 for late orders — and the relationship is monotonic across delay buckets: scores fall steadily from 4.32 (early by >7 days) down to 1.70 (8+ days late). This is a strong, consistent association in observational data; it does **not** prove lateness *causes* the lower score (dissatisfied customers may also be more likely to leave a review, or a delay may coincide with other problems like a damaged item), but the pattern is large enough to be a genuine operational signal.

## Geography

In [11]:
state_summary = (
    orders_analytics[orders_analytics["is_delivered"]]
    .groupby("customer_state")
    .agg(revenue=("item_revenue", "sum"), orders=("order_id", "nunique"),
         customers=("customer_unique_id", "nunique"), avg_review=("avg_review_score", "mean"))
)
state_summary["aov"] = state_summary["revenue"] / state_summary["orders"]
state_summary = state_summary.sort_values("revenue", ascending=False)

fig = px.bar(state_summary.reset_index().head(10), x="customer_state", y="revenue",
             title="Top 10 states by delivered-order revenue")
fig.show()
state_summary.head(10).round(2)

,revenue,orders,customers,avg_review,aov
customer_state,,,,,
SP,5067633.16,40501,39156,4.25,125.12
RJ,1759651.13,12350,11917,3.97,142.48
MG,1552481.83,11354,11001,4.19,136.73
RS,728897.47,5345,5168,4.19,136.37
PR,666063.51,4923,4769,4.24,135.30
SC,507012.13,3546,3449,4.13,142.98
BA,493584.14,3256,3158,3.93,151.59
DF,296498.41,2080,2019,4.13,142.55
GO,282836.70,1957,1895,4.10,144.53


**Small-sample caution:** the smallest states (RR, AP, AC — each under 100 delivered orders) have wide statistical uncertainty in their averages; their `avg_review`/`aov` figures should be treated as indicative, not precise, and are not used for state-level conclusions below.

**Finding:** revenue is heavily concentrated in São Paulo (SP), which alone accounts for ~38% of total revenue and has the lowest AOV (R$125) and best on-time delivery performance of any major state (from `sql/04_delivery_analysis.sql`) — consistent with SP being both the largest customer base and the shortest average shipping distance from sellers, most of whom are also based there. Rio de Janeiro (RJ) stands out with a notably lower average review score (3.97) and on-time rate (87.9%) than other high-volume states.

## Initial Findings

Answering the five Phase 1 business questions with the calculations above:

1. **How much revenue was generated from delivered orders?**
   R$13,221,498.11 across 96,478 delivered orders (AOV R$137.04).

2. **How did revenue and order volume change over time?**
   Revenue grew steadily from platform launch (late 2016) through late 2017, then plateaued around R$0.85–0.98M/month through mid-2018. Growth is driven by order volume, not AOV, which stayed roughly flat (R$124–152).

3. **Which product categories generated the most delivered-order revenue?**
   `health_beauty` (9.3% of revenue), `watches_gifts` (8.8%), and `bed_bath_table` (7.7%) lead, but revenue is broadly spread — no category exceeds 10% of the total.

4. **What percentage of customers purchased more than once?**
   Only 3.0% of customers (2,801 of 93,358) placed a second delivered order, contributing just 5.5% of total revenue. Repeat purchasing is genuinely low in this dataset, not an artifact of measurement.

5. **Are late deliveries associated with worse review scores?**
   Yes, strongly. On-time/early orders average 4.29/5 vs. 2.27/5 for late orders, falling monotonically as delay increases. This is an association observed in the data, not a proven causal effect.

Business recommendations based on these findings are deferred to a later phase.